# 14 · Cross-dataset generalisation (benign vs attack)

Trains on **NF-ToN-IoT-v2** and tests on **NF-UNSW-NB15-v3**, and the reverse, using only the NetFlow features both datasets share (same column names) and the binary label.
Same leakage-safe feature policy and the same fixed model configuration; in-domain scores on the same shared features are the reference.
The two datasets come from different testbeds, attack tools and traffic mixes. Per-attack-type detection on the target dataset shows which attacks transfer.

Needs both CSVs (`NF-ToN-IoT-v2.csv`, `NF-UNSW-NB15-v3.csv`). Run once (no `ACTIVE_DATASET` needed).

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from eval_helpers import *
import feature_policy as fp, generalization as gz
import time
PAIRS = [("nf_ton_iot_v2_binary", "nf_unsw_nb15_v3_binary"), ("nf_unsw_nb15_v3_binary", "nf_ton_iot_v2_binary")]
PROF = DATASET_PROFILES
for p in {x for pr in PAIRS for x in pr}: assert Path(PROF[p]["path"]).exists(), f"CSV missing: {PROF[p]['path']}"
src_meta = json.load(open(ROOT / "configs" / "nf_ton_iot_v2_fixed_params.json", encoding="utf-8")); BASE_PARAMS = dict(src_meta["best_params"])
print("train rows:", f"{GEN_TRAIN_ROWS:,}", "| test rows cap:", f"{GEN_EVAL_ROWS:,}", "| model:", BASE_PARAMS)

## 1 · Shared features and partitions

In [ ]:
def prep(p):
    prof = PROF[p]; path = prof["path"]; header = pd.read_csv(path, nrows=0).columns.tolist()
    g = fp.classify_columns(header, prof["target_col"], prof["drop_cols"], prof["ip_cols"]); core = fp.numeric_columns(path, g["core"])
    le, y = fp.read_target(path, prof["target_col"]); part = fp.make_partition(y, GEN_TRAIN_ROWS, GEN_EVAL_ROWS, SEED)      # exact pool size, random sample
    return {"path": path, "core": core, "y": y, "part": part, "attack_col": prof.get("attack_col"), "classes": list(le.classes_)}
D = {p: prep(p) for p in {x for pr in PAIRS for x in pr}}
for p, d in D.items(): print(f"{p}: {len(d['y']):,} rows | {len(d['core'])} features | attack share {d['y'].mean():.3f}")
for s, t in PAIRS:
    common = sorted(set(D[s]["core"]) & set(D[t]["core"])); print(f"{s} -> {t}: {len(common)} shared features")

## 2 · Train on the source, test in-domain and on the other dataset

In [ ]:
rows, atk_rows = [], []
for s, t in PAIRS:
    common = sorted(set(D[s]["core"]) & set(D[t]["core"]))
    if len(common) < 5: print(f"{s} -> {t}: only {len(common)} shared features -> skipped"); continue
    ds_, dt_ = fp.read_rows(D[s]["path"], common, D[s]["part"]), fp.read_rows(D[t]["path"], common, D[t]["part"])
    Xp, yp = ds_["pool"], D[s]["y"][ds_["pool"].index.values]
    K = min(30, len(common)); rare, xrare = infer_rare_classes(yp)
    for mname, clf in [("fixed configuration", DecisionTreeClassifier(random_state=SEED, **BASE_PARAMS)), ("default tree", DecisionTreeClassifier(random_state=SEED))]:
        pipe = build_safe_pipeline(clf, k=K); pipe.fit(Xp, yp)
        for where, Xe_, ye_ in [("in-domain", ds_["test"], D[s]["y"][ds_["test"].index.values]), ("cross-dataset", dt_["test"], D[t]["y"][dt_["test"].index.values])]:
            pr = pipe.predict(Xe_); m = compute_enhanced_metrics(ye_, pr, rare, xrare); tn, fp_, fn, tp = confusion_matrix(ye_, pr, labels=[0, 1]).ravel()
            rows.append({"train": s.replace("_binary", ""), "test": (s if where == "in-domain" else t).replace("_binary", ""), "where": where, "model": mname, "shared_features": len(common), "train_rows": len(Xp), "test_rows": len(Xe_),
                         "mcc": m["mcc"], "accuracy": m["accuracy"], "attack_recall": tp / max(tp + fn, 1), "false_positive_rate": fp_ / max(fp_ + tn, 1), "attack_precision": tp / max(tp + fp_, 1), "nodes": int(pipe.named_steps["clf"].tree_.node_count)})
            print(f"{s.replace('_binary',''):16s} -> {rows[-1]['test']:16s} [{where:13s}] {mname:20s} MCC {m['mcc']:7.4f}  attack recall {rows[-1]['attack_recall']:.3f}  FPR {rows[-1]['false_positive_rate']:.3f}", flush=True)
            if where == "cross-dataset" and mname == "fixed configuration" and D[t]["attack_col"]:
                atk = pd.read_csv(D[t]["path"], usecols=[D[t]["attack_col"]], dtype={D[t]["attack_col"]: "category"})[D[t]["attack_col"]].astype(str)
                at = atk.loc[Xe_.index].to_numpy(); pm = pr == 1
                for a_ in sorted(set(at)):
                    mm = at == a_; atk_rows.append({"train": s.replace("_binary", ""), "test": t.replace("_binary", ""), "attack_type": a_, "n": int(mm.sum()), "flagged_malicious_rate": float(pm[mm].mean())})
    del ds_, dt_
res = pd.DataFrame(rows); OUT = Path("generalization") / "cross_dataset"; OUT.mkdir(parents=True, exist_ok=True)
res.to_csv(OUT / "cross_dataset.csv", index=False); pd.DataFrame(atk_rows).to_csv(OUT / "cross_dataset_attack_types.csv", index=False); display(res.round(4))

In [ ]:
fx = res[res.model == "fixed configuration"] if "model" in res.columns else res.iloc[0:0]
if len(fx):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True); lab = lambda r: f"{r['train']}\n-> {r['test']}"
    for ax, (m, ttl) in zip(axes, [("mcc", "MCC"), ("attack_recall", "attack recall")]):
        for i, (_, r) in enumerate(fx.iterrows()):
            ax.bar(i, r[m], color="#1f77b4" if r["where"] == "in-domain" else "#d95f02"); ax.text(i, r[m] + .01, f"{r[m]:.2f}", ha="center", fontsize=8)
        ax.set_xticks(range(len(fx)), [lab(r) for _, r in fx.iterrows()], fontsize=7); ax.set_ylim(0, 1.1); ax.set_title(f"{ttl}  (blue = in-domain, orange = other dataset)")
    fig.savefig(FIGURES_DIR / "cross_dataset.png", dpi=300, bbox_inches="tight"); plt.show()
    print()
    for tr, g in fx.groupby("train"):
        a, b = g[g["where"] == "in-domain"].iloc[0], g[g["where"] == "cross-dataset"].iloc[0]
        print(f"trained on {tr}: in-domain MCC {a.mcc:.3f} -> other dataset MCC {b.mcc:.3f} (drop {a.mcc - b.mcc:+.3f}); attack recall {a.attack_recall:.3f} -> {b.attack_recall:.3f}; false-positive rate {a.false_positive_rate:.3f} -> {b.false_positive_rate:.3f}")
    print("\nThe two datasets come from different testbeds; a drop means the model must be trained per deployment network.")